In [5]:

import pandas as pd
import numpy as np
from pathlib import Path

# Check required project files
required_files = [
    "../data/processed/reviews_cleaned.csv",
    "../data/processed/accommodation_cleaned.csv",
    "../data/processed/activities_cleaned.csv",
    "../data/processed/water_sports_cleaned.csv",
    "../data/processed/destination_master.csv",
    "../data/processed/destination_features.csv",
    "../data/processed/hotel_master.csv",
    "../data/processed/destination_evaluation_summary.csv"
]

for file in required_files:
    path = Path(file)
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status}: {file}")


FOUND: ../data/processed/reviews_cleaned.csv
FOUND: ../data/processed/accommodation_cleaned.csv
FOUND: ../data/processed/activities_cleaned.csv
FOUND: ../data/processed/water_sports_cleaned.csv
FOUND: ../data/processed/destination_master.csv
FOUND: ../data/processed/destination_features.csv
FOUND: ../data/processed/hotel_master.csv
FOUND: ../data/processed/destination_evaluation_summary.csv


In [6]:

destination_features = pd.read_csv(
    "../data/processed/destination_features.csv"
)

def final_destination_test(
    preferred_type="Beaches",
    min_rating=4.0,
    top_n=5
):
    results = destination_features.copy()

    results = results[
        results["Location_Type"].str.casefold()
        == preferred_type.casefold()
    ]

    results = results[
        results["Average_Rating"] >= min_rating
    ]

    results["Recommendation_Score"] = (
        0.7 * results["Rating_Score"]
        + 0.3 * results["Review_Popularity"]
    )

    results = results.sort_values(
        "Recommendation_Score",
        ascending=False
    ).head(top_n).reset_index(drop=True)

    return results

final_destinations = final_destination_test()

display(
    final_destinations[
        [
            "Location_Name",
            "Location_Type",
            "Average_Rating",
            "Recommendation_Score"
        ]
    ]
)

print("Recommendation count:", len(final_destinations))
print(
    "Duplicate destinations:",
    final_destinations["Location_Name"].duplicated().sum()
)
print(
    "All scores between 0 and 1:",
    final_destinations["Recommendation_Score"].between(0, 1).all()
)


,Location_Name,Location_Type,Average_Rating,Recommendation_Score
0,Bentota Beach,Beaches,4.59,0.817681
1,Passikudah Beach,Beaches,4.56,0.753081
2,Mirissa Beach,Beaches,4.27,0.748673
3,Nilaveli Beach,Beaches,4.33,0.716896
4,Hikkaduwa Beach,Beaches,4.02,0.701671


Recommendation count: 5
Duplicate destinations: 0
All scores between 0 and 1: True


In [7]:

hotel_master = pd.read_csv(
    "../data/processed/hotel_master.csv"
)

destination_master = pd.read_csv(
    "../data/processed/destination_master.csv"
)

def final_hotel_test(destination_name, top_n=5):
    destination = destination_master[
        destination_master["Location_Name"].str.casefold()
        == destination_name.strip().casefold()
    ]

    if destination.empty:
        return pd.DataFrame()

    district = destination.iloc[0]["Mapped_District"]

    if pd.isna(district):
        return pd.DataFrame()

    results = hotel_master[
        hotel_master["District_Key"].str.casefold()
        == str(district).strip().casefold()
    ].copy()

    results = results.drop_duplicates(subset=["Name"])

    return results[
        ["Name", "Type", "District", "Address"]
    ].head(top_n).reset_index(drop=True)


final_hotels = final_hotel_test("Bentota Beach")

display(final_hotels)

print("Hotel recommendations:", len(final_hotels))
print("Duplicate hotel names:", final_hotels["Name"].duplicated().sum())
print("Missing hotel names:", final_hotels["Name"].isna().sum())


,Name,Type,District,Address
0,GALLE FORT HOTEL,Boutique Hotels,Galle,"NO.28, CHURCH STREET, FORT , GALLE"
1,ADITYA,Boutique Hotels,Galle,"719/1, GALLE ROAD, DEVENIGODA,RATHGAMA"
2,CLUB VILLA,Boutique Hotels,Galle,"NO.138/15, GALLE ROAD, BENTOTA"
3,SAMAN VILLAS,Boutique Hotels,Galle,"ATURUWELLA, INDURUWA"
4,TAMARIND HILL HOTEL,Boutique Hotels,Galle,"NO.288,GALLE ROAD, DADELLA, GALLE"


Hotel recommendations: 5
Duplicate hotel names: 0
Missing hotel names: 0


In [11]:
from pathlib import Path
import importlib.util

# Locate the project folder
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

hotel_recommender_path = (
    PROJECT_ROOT / "app" / "hotel_recommender.py"
)

print("Current folder:", current_path)
print("Project root:", PROJECT_ROOT)
print("Recommender file exists:", hotel_recommender_path.exists())

if not hotel_recommender_path.exists():
    raise FileNotFoundError(
        f"Cannot find hotel_recommender.py at: {hotel_recommender_path}"
    )

spec = importlib.util.spec_from_file_location(
    "hotel_recommender",
    hotel_recommender_path
)
hotel_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hotel_module)

recommend_hotels = hotel_module.recommend_hotels

print("Hotel recommender imported successfully!")

Current folder: c:\Users\chami\OneDrive\Desktop\Tourist_Recommendation_System\notebooks
Project root: c:\Users\chami\OneDrive\Desktop\Tourist_Recommendation_System
Recommender file exists: True
Hotel recommender imported successfully!


In [12]:
bentota_results = recommend_hotels(
    destination_name="Bentota Beach",
    top_n=10
)

address_match = bentota_results["Address"].fillna("").str.contains(
    "Bentota", case=False, na=False
)

print("Total results:", len(bentota_results))
print("Addresses mentioning Bentota:", int(address_match.sum()))

assert len(bentota_results) > 0
assert address_match.all()

print("PASS: All returned addresses mention Bentota.")

Total results: 10
Addresses mentioning Bentota: 10
PASS: All returned addresses mention Bentota.


In [13]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PATH = PROJECT_ROOT / "data" / "processed"

destination_features = pd.read_csv(
    DATA_PATH / "destination_features.csv"
)

print("Dataset shape:", destination_features.shape)
print("Missing destination names:",
      destination_features["Location_Name"].isna().sum())
print("Duplicate destination names:",
      destination_features["Location_Name"].duplicated().sum())
print("Missing average ratings:",
      destination_features["Average_Rating"].isna().sum())

assert destination_features["Location_Name"].notna().all()
assert not destination_features["Location_Name"].duplicated().any()

print("\nPASS: Destination names are complete and unique.")


Dataset shape: (76, 15)
Missing destination names: 0
Duplicate destination names: 0
Missing average ratings: 0

PASS: Destination names are complete and unique.


In [14]:

# Validate rating values and recommendation scores

print("Average rating range:")
print("Minimum:", destination_features["Average_Rating"].min())
print("Maximum:", destination_features["Average_Rating"].max())

rating_valid = destination_features["Average_Rating"].between(1, 5).all()
print("\nAll ratings between 1 and 5:", rating_valid)

score_columns = ["Rating_Score", "Review_Popularity"]
for column in score_columns:
    if column in destination_features.columns:
        print(
            f"{column} range:",
            destination_features[column].min(),
            "to",
            destination_features[column].max()
        )
        print(
            f"{column} within 0-1:",
            destination_features[column].between(0, 1).all()
        )

assert rating_valid, "Some average ratings are outside 1-5."

print("\nPASS: All average ratings are within the expected range.")


Average rating range:
Minimum: 3.12
Maximum: 4.99

All ratings between 1 and 5: True
Rating_Score range: 0.53 to 0.9975
Rating_Score within 0-1: True
Review_Popularity range: 0.0948509485094851 to 1.0
Review_Popularity within 0-1: True

PASS: All average ratings are within the expected range.


In [16]:

import pandas as pd

# Test general recommendations
general_results = recommend_destinations(
    preferred_type="All",
    min_rating=4.0,
    top_n=5
)

# Convert list of dictionaries into a DataFrame
general_results = pd.DataFrame(general_results)

print("General recommendation count:", len(general_results))
display(general_results)

# Validate results
assert len(general_results) <= 5
assert general_results["Average_Rating"].ge(4.0).all()
assert general_results["Location_Name"].notna().all()
assert not general_results["Location_Name"].duplicated().any()

print("PASS: General recommendation checks passed.")


General recommendation count: 5


,Location_Name,Located_City,Location_Type,Average_Rating,Review_Count,Most_Common_Travel_Month,Destination_District,Destination_Province,Mapped_District,Geography_Notes,District_Key,Accommodation_Count,Rating_Score,Review_Popularity,Travel_Month_Category,Recommendation_Score
0,Royal Botanical Gardens,Peradeniya,Gardens,4.57,337,1,Kandy,Central,Kandy,NaN,kandy,266,0.8925,0.913279,1,0.898734
1,Community Tsunami Museum,Hikkaduwa,Museums,4.67,302,1,Galle,Southern,Galle,NaN,galle,284,0.9175,0.818428,1,0.887778
2,Handunugoda Tea Estate,Ahangama,Farms,4.64,304,1,Galle,Southern,Galle,NaN,galle,284,0.9100,0.823848,1,0.884154
3,Sigiriya The Ancient Rock Fortress,Sigiriya,Historic Sites,4.65,285,1,NaN,Central,Matale,NaN,matale,86,0.9125,0.772358,1,0.870457
4,Polonnaruwa,Polonnaruwa,Historic Sites,4.42,333,3,NaN,North Central,Polonnaruwa,NaN,polonnaruwa,35,0.8550,0.902439,3,0.869232


PASS: General recommendation checks passed.


In [17]:

# Test accommodation recommendations for Bentota Beach
from pathlib import Path
import importlib.util
import pandas as pd

hotel_path = PROJECT_ROOT / "app" / "hotel_recommender.py"

spec = importlib.util.spec_from_file_location(
    "hotel_recommender_test",
    hotel_path
)
hotel_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hotel_module)

recommend_hotels = hotel_module.recommend_hotels

hotel_results = recommend_hotels(
    destination_name="Bentota Beach",
    top_n=10
)

print("Accommodation recommendation count:", len(hotel_results))
print("Missing accommodation names:",
      hotel_results["Name"].isna().sum())
print("Duplicate accommodation names:",
      hotel_results["Name"].duplicated().sum())

assert len(hotel_results) <= 10
assert hotel_results["Name"].notna().all()
assert not hotel_results["Name"].duplicated().any()

print("PASS: Accommodation recommendation checks passed.")


Accommodation recommendation count: 10
Missing accommodation names: 0
Duplicate accommodation names: 0
PASS: Accommodation recommendation checks passed.


In [18]:

# Final testing summary

test_summary = pd.DataFrame([
    {
        "Test": "Destination dataset quality",
        "Result": "PASS",
        "Details": "76 destinations; no missing or duplicate names"
    },
    {
        "Test": "Rating and score ranges",
        "Result": "PASS",
        "Details": "Ratings within 1-5; scores within 0-1"
    },
    {
        "Test": "Destination recommendations",
        "Result": "PASS",
        "Details": "5 results; rating filter and uniqueness checks passed"
    },
    {
        "Test": "Accommodation recommendations",
        "Result": "PASS",
        "Details": "10 results; no missing or duplicate names"
    },
    {
        "Test": "Missing district mappings",
        "Result": "PASS",
        "Details": "3 unmapped destinations returned no accommodation"
    },
    {
        "Test": "Bentota address matching",
        "Result": "PASS",
        "Details": "10 of 10 returned addresses mention Bentota"
    }
])

display(test_summary)

print("\nTotal tests:", len(test_summary))
print("Passed:", (test_summary["Result"] == "PASS").sum())


,Test,Result,Details
0,Destination dataset quality,PASS,76 destinations; no missing or duplicate names
1,Rating and score ranges,PASS,Ratings within 1-5; scores within 0-1
2,Destination recommendations,PASS,5 results; rating filter and uniqueness checks...
3,Accommodation recommendations,PASS,10 results; no missing or duplicate names
4,Missing district mappings,PASS,3 unmapped destinations returned no accommodation
5,Bentota address matching,PASS,10 of 10 returned addresses mention Bentota



Total tests: 6
Passed: 6
